## IMPORT & UPLOAD


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import io
from google.colab import files

In [ ]:
print("VUI LÒNG UPLOAD TẤT CẢ CÁC FILE CSV")
# Dictionary để lưu trữ tất cả các dataframe
dataframes = {}
# Danh sách các file cần upload
files_to_upload = [
    'orders.csv',
    'products.csv',
    'returns.csv',
    'web_traffic.csv',
    'order_items.csv',
    'customers.csv',
    'geography.csv',
    'payments.csv'
]
# Upload tất cả các file
for file_name in files_to_upload:
    print(f"\n📁 Đang upload file: {file_name}")
    uploaded = files.upload()

    for fn in uploaded.keys():
        print(f'✅ Đã upload file "{fn}" với dung lượng {len(uploaded[fn])} bytes')
        # Đọc file vào dataframe
        df = pd.read_csv(io.StringIO(uploaded[fn].decode('utf-8')))
        # Lưu vào dictionary với key là tên file (bỏ phần .csv)
        key_name = file_name.replace('.csv', '')
        dataframes[key_name] = df
        print(f'📊 Đã đọc thành công {len(df)} dòng dữ liệu')
        break


VUI LÒNG UPLOAD TẤT CẢ CÁC FILE CSV

📁 Đang upload file: orders.csv


KeyboardInterrupt: 

In [ ]:
# Gán các dataframe vào biến riêng
orders_df = dataframes['orders']
products_df = dataframes['products']
returns_df = dataframes['returns']
web_traffic_df = dataframes['web_traffic']
order_items_df = dataframes['order_items']
customers_df = dataframes['customers']
geography_df = dataframes['geography']
payments_df = dataframes['payments']

#QUESTION_01

In [ ]:
df = orders_df.copy()
print('--- DataFrame Info (Data Types and Non-Null Counts) ---')
df.info()
print('\n--- Number of Duplicate Rows ---')
print(df.duplicated().sum())
print('\n--- Missing Values per Column ---')
print(df.isnull().sum())

In [ ]:
# Convert 'order_date' to datetime objects
df['order_date'] = pd.to_datetime(df['order_date'])
print("Converted 'order_date' to datetime.")

In [ ]:
# Sort by customer_id and order_date to ensure correct sequence for diff()
cust_orders = df.sort_values(by=['customer_id', 'order_date']).copy()

# Calculate the difference between consecutive order dates for each customer
cust_orders['inter_order_gap'] = cust_orders.groupby('customer_id')['order_date'].diff()

print("Calculated inter-order gaps for each customer.")

In [ ]:
# Filter for customers with more than one order (i.e., those with a non-null inter_order_gap)
customers_with_multiple_orders = cust_orders.dropna(subset=['inter_order_gap'])

# Calculate the median of the inter-order gaps
median_inter_order_gap = customers_with_multiple_orders['inter_order_gap'].median()

print(f"Trung vị số ngày giữa hai lần mua liên tiếp đối với khách hàng có nhiều hơn một đơn hàng là: {median_inter_order_gap}\n")

## Question_02


In [ ]:
df = products_df.copy()
df.head()
print('--- DataFrame Info (Data Types and Non-Null Counts) ---')
df.info()
print('\n--- Number of Duplicate Rows ---')
print(df.duplicated().sum())
print('\n--- Missing Values per Column ---')
print(df.isnull().sum())

In [ ]:
df['tỷ suất lợi nhuận gộp'] = (df['price'] - df['cogs']) / df['price']
df.head()

In [ ]:
# Group by 'segment' and calculate the mean of 'tỷ suất lợi nhuận gộp'
average_gross_margin_by_segment = df.groupby('segment')['tỷ suất lợi nhuận gộp'].mean()

# Find the segment with the highest average gross margin
best_segment = average_gross_margin_by_segment.idxmax()
max_gross_margin = average_gross_margin_by_segment.max()

print(f"Phân khúc sản phẩm có tỷ suất lợi nhuận gộp trung bình cao nhất là: {best_segment}")
print(f"Tỷ suất lợi nhuận gộp trung bình cao nhất: {max_gross_margin:.4f}\n")

## QUESTION_03

In [ ]:
df = returns_df.copy()
df.head()
print('--- DataFrame Info (Data Types and Non-Null Counts) ---')
df.info()
print('\n--- Number of Duplicate Rows ---')
print(df.duplicated().sum())
print('\n--- Missing Values per Column ---')
print(df.isnull().sum())


In [ ]:
# Filter products for 'Streetwear' category
streetwear_products = products_df[products_df['category'] == 'Streetwear']

# Merge returns data with streetwear products data
merged_df = pd.merge(df, streetwear_products, on='product_id', how='inner')

if merged_df.empty:
    print("Không tìm thấy bản ghi trả hàng nào liên kết với sản phẩm thuộc danh mục Streetwear.")
elif merged_df['return_reason'].empty:
    print("Không có lý do trả hàng nào trong các bản ghi trả hàng của sản phẩm Streetwear.")
else:
    # Find the most frequent return reason
    mode_results = merged_df['return_reason'].mode()
    if not mode_results.empty:
        most_common_return_reason = mode_results[0]
        print(f"Lý do trả hàng xuất hiện nhiều nhất cho sản phẩm thuộc danh mục Streetwear là: {most_common_return_reason}\n")
    else:
        print("Không tìm thấy lý do trả hàng phổ biến nhất cho sản phẩm Streetwear.\n")

#QUESTION_04

In [ ]:
df = web_traffic_df.copy()
df.head()
print('--- DataFrame Info (Data Types and Non-Null Counts) ---')
df.info()
print('\n--- Number of Duplicate Rows ---')
print(df.duplicated().sum())
print('\n--- Missing Values per Column ---')
print(df.isnull().sum())

In [ ]:
average_bounce_rate_by_source = df.groupby('traffic_source')['bounce_rate'].mean()

print("Tỷ lệ thoát trung bình theo nguồn truy cập:")
print(average_bounce_rate_by_source)
print("\n")

#QUESTION_05


In [ ]:
df = order_items_df.copy()
df.head()
print('--- DataFrame Info (Data Types and Non-Null Counts) ---')
df.info()
print('\n--- Number of Duplicate Rows ---')
print(df.duplicated().sum())
print('\n--- Missing Values per Column ---')
print(df.isnull().sum())

In [ ]:
total_rows = len(df)
# Count rows where promo_id is not null
promoted_items_count = df['promo_id'].count()

# Calculate the percentage
percentage_promoted = (promoted_items_count / total_rows) * 100

print(f"Tỷ lệ phần trăm các dòng có áp dụng khuyến mãi là: {percentage_promoted:.2f}%\n")

#QUESTION_06


In [ ]:
df = customers_df.copy()
df.head()
print('--- DataFrame Info (Data Types and Non-Null Counts) ---')
df.info()
print('\n--- Number of Duplicate Rows ---')
print(df.duplicated().sum())
print('\n--- Missing Values per Column ---')
print(df.isnull().sum())


In [ ]:
# Merge orders with customer information on 'customer_id'
merged_orders_customers = pd.merge(orders_df, customers_df, on='customer_id', how='inner')

# Filter out null age_groups
filtered_df = merged_orders_customers.dropna(subset=['age_group'])

# Group by age_group to calculate total orders and unique customers
orders_per_age_group = filtered_df.groupby('age_group').agg(
    total_orders=('order_id', 'nunique'),
    unique_customers=('customer_id', 'nunique')
).reset_index()

# Calculate average orders per customer for each age group
orders_per_age_group['average_orders_per_customer'] = orders_per_age_group['total_orders'] / orders_per_age_group['unique_customers']

# Find the age group with the highest average orders per customer
best_age_group = orders_per_age_group.loc[orders_per_age_group['average_orders_per_customer'].idxmax()]

print(f"Nhóm tuổi có số đơn hàng trung bình trên mỗi khách hàng cao nhất là: {best_age_group['age_group']}")
print(f"Với số đơn hàng trung bình: {best_age_group['average_orders_per_customer']:.2f}\n")


#QUESTION_07

In [ ]:
df = geography_df.copy()
df.head()
print('--- DataFrame Info (Data Types and Non-Null Counts) ---')
df.info()
print('\n--- Number of Duplicate Rows ---')
print(df.duplicated().sum())
print('\n--- Missing Values per Column ---')
print(df.isnull().sum())

In [ ]:
# Calculate total revenue per item
order_items_df['total_revenue_item'] = order_items_df['quantity'] * order_items_df['unit_price']

# Merge order_items_df with orders_df to link order items to customer zip codes
order_revenue_with_zip = pd.merge(order_items_df, orders_df[['order_id', 'zip']], on='order_id', how='left')

# Merge with geography_df to link zip codes to regions
final_revenue_df = pd.merge(order_revenue_with_zip, geography_df[['zip', 'region']], on='zip', how='left')

# Group by region and sum the total revenue
revenue_by_region = final_revenue_df.groupby('region')['total_revenue_item'].sum().reset_index()

# Find the region with the highest total revenue
top_region = revenue_by_region.loc[revenue_by_region['total_revenue_item'].idxmax()]

print(f"Vùng tạo ra tổng doanh thu cao nhất là: {top_region['region']}")
print(f"Với tổng doanh thu: {top_region['total_revenue_item']:.2f}\n")

#QUESTION_08

In [ ]:
cancelled_orders = orders_df[orders_df['order_status'] == 'cancelled']
print("Các đơn hàng có trạng thái 'cancelled':")
print(cancelled_orders.head())

In [ ]:
payment_method_counts = orders_df['payment_method'].value_counts()
print("\nThống kê các phương thức thanh toán:")
print(payment_method_counts)
print("\n")

#QUESTION_09

In [ ]:
# Merge returns_df with products_df to get size for returned items
returned_items_with_size = pd.merge(returns_df, products_df[['product_id', 'size']], on='product_id', how='left')

# Count the number of returned items per size
returned_counts_by_size = returned_items_with_size.groupby('size').size().reset_index(name='returned_count')

print("Returned items by size:")
print(returned_counts_by_size)

In [ ]:
# Merge order_items_df with products_df to get size for all ordered items
ordered_items_with_size = pd.merge(order_items_df, products_df[['product_id', 'size']], on='product_id', how='left')

# Count the number of ordered items per size
ordered_counts_by_size = ordered_items_with_size.groupby('size').size().reset_index(name='ordered_count')

print("\nOrdered items by size:")
print(ordered_counts_by_size)

In [ ]:
# Merge the returned_counts_by_size and ordered_counts_by_size dataframes
size_return_analysis = pd.merge(returned_counts_by_size, ordered_counts_by_size, on='size', how='inner')

# Calculate the return rate
size_return_analysis['return_rate'] = size_return_analysis['returned_count'] / size_return_analysis['ordered_count']

# Sort by return rate to find the highest
highest_return_rate_size = size_return_analysis.sort_values(by='return_rate', ascending=False).iloc[0]

print("\nTỷ lệ trả hàng theo kích thước:")
print(size_return_analysis)
print(f"\n✅ Kích thước có tỷ lệ trả hàng cao nhất là: {highest_return_rate_size['size']} với tỷ lệ {highest_return_rate_size['return_rate']:.4f}\n")


#QUESTION_10


In [ ]:
# Calculate the total payment for each order_id
total_payment_per_order = payments_df.groupby('order_id')['payment_value'].sum().reset_index()

# Merge this back with the original payments_df to get installments for each order's total payment
merged_payments = pd.merge(total_payment_per_order, payments_df[['order_id', 'installments']].drop_duplicates(), on='order_id', how='left')

# Calculate the average payment value for each installment
average_payment_by_plan = merged_payments.groupby('installments')['payment_value'].mean().reset_index()

# Find the installment with the highest average payment value
highest_avg_payment_plan = average_payment_by_plan.loc[average_payment_by_plan['payment_value'].idxmax()]

print(f"Kế hoạch trả góp có giá trị thanh toán trung bình trên mỗi đơn hàng cao nhất là: {highest_avg_payment_plan['installments']} (trả góp).")
print(f"Giá trị thanh toán trung bình cao nhất là: {highest_avg_payment_plan['payment_value']:.2f}")
